In [0]:
from pyspark.sql import functions as F

In [0]:
market_watchlist_historical = spark.sql('''
          SELECT *,
          ROW_NUMBER() OVER (PARTITION BY symbol, email ORDER BY _timestamp DESC) AS Row_Ct
          FROM `dataexpert-portfolio`.`market-data-tracker`.lb_watchlist_history
          ORDER BY symbol, email, _timestamp DESC
          ''')

market_watchlist_historical.show()

In [0]:
scd_new_schema = market_watchlist_historical.withColumn("tickerxuser_key", F.concat(F.col("symbol"), F.lit("_"), F.col("email")))

scd_new_schema = scd_new_schema.withColumn("is_active", F.when(F.col("Row_Ct") == 1, True).otherwise(False))

scd_new_schema = scd_new_schema.select("symbol",F.col("latest_price").alias("price"), "email" ,F.col("_pg_change_type").alias("change_type"), F.col("_pg_xid").alias("id"),"_sort_by", "is_active","tickerxuser_key" ,F.col("updated_at").alias("start_date") )

scd_new_schema.show()

In [0]:
%sql
WITH main_table AS (
SELECT
`_pg_xid` as id,
symbol,
latest_price,
email,
concat(symbol,"_",email) as tickerxuser_key,
updated_at, 
ROW_NUMBER() OVER (PARTITION BY symbol, email ORDER BY updated_at DESC) AS row_num
FROM `dataexpert-portfolio`.`market-data-tracker`.lb_watchlist_history
),

prev_rank_updated AS (
    
)